# Les 11 - Agent-tot-Agent (A2A) Protocol


## Installatie


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity

In [ ]:
import os
import asyncio
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Wat is het A2A-protocol?

Het **Agent-naar-agent (A2A) protocol** is een open standaard die AI-agenten in staat stelt om te communiceren,
elkaar te ontdekken en samen te werken — zelfs wanneer ze zijn gebouwd op verschillende frameworks of worden gehost
door verschillende diensten.

Belangrijke concepten:

- **Ontdekking** – Agenten publiceren een *Agent Card* die hun mogelijkheden beschrijft, waardoor het
  voor andere agenten (of orchestratoren) eenvoudig is om de juiste specialist voor een taak te vinden.
- **Berichtenuitwisseling** – Agenten wisselen gestructureerde berichten uit via een gemeenschappelijk protocol, zodat een
  verzoek van de ene agent door een andere begrepen en uitgevoerd kan worden, ongeacht de interne
  implementatie.
- **Taaklevenscyclus** – A2A definieert toestanden zoals *ingediend*, *bezig*, *voltooid*, en
  *mislukt*, waardoor de orchestrator volledig zicht heeft op hoe een gedelegeerde taak vordert.

In deze les simuleren we A2A-stijl samenwerking door drie gespecialiseerde reisagenten
in een workflow te koppelen waarbij elke agent zijn expertise bijdraagt en resultaten doorgeeft aan de volgende.


## Het creëren van gespecialiseerde reisagenten


In [ ]:
currency_agent = await provider.create_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = await provider.create_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = await provider.create_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## Samenwerking tussen meerdere agenten via workflow

We verbinden de drie agenten in een sequentiële workflow die A2A-berichtuitwisseling weerspiegelt:

1. **CurrencyExchangeAgent** ontvangt het gebruikersverzoek en geeft valuta-advies.
2. **ActivityPlannerAgent** ontvangt de verrijkte context en voegt aanbevelingen voor activiteiten toe.
3. **TravelManagerAgent** combineert beide invoergegevens tot een definitief reisoverzicht.


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## A2A in productie begrijpen

In een productieomgeving ontgrendelt het A2A-protocol krachtige cross-service scenario's:

| Mogelijkheid | Beschrijving |
|---|---|
| **Cross-framework interoperabiliteit** | Een agent die met één framework is gebouwd kan taken delegeren aan een agent die met elk ander A2A-compatibel framework is gebouwd, waardoor echte interoperabiliteit tussen organisaties mogelijk wordt. |
| **Servicegrenzen** | Agenten kunnen in afzonderlijke microservices, cloudregio's of zelfs verschillende organisaties worden ondergebracht en toch naadloos samenwerken. |
| **Dynamische ontdekking** | Een orchestrator kan tijdens runtime een Agent Card-register raadplegen om de meest geschikte specialist voor een bepaalde subtaak te vinden. |
| **Streaming & pushmeldingen** | A2A ondersteunt Server-Sent Events (SSE) voor realtime voortgangsupdates en pushmeldingen voor langlopende taken. |

De workflow die we hierboven hebben opgebouwd is een vereenvoudigde, in-process versie van dit patroon. In een echte
implementatie zou elke agent een HTTP-endpoint blootstellen, een Agent Card publiceren en communiceren
via het A2A JSON-RPC-protocol.


## Samenvatting

In deze les heb je geleerd:

1. **Wat het A2A-protocol is** — een open standaard voor agent-naar-agentontdekking, berichtgeving,
   en taakbeheer.
2. **Hoe je gespecialiseerde agenten maakt** — een Valutawissel-agent, een Activiteitenplanner-agent,
   en een Reismanager-orchestrator.
3. **Hoe je agenten in een workflow koppelt** — met behulp van `WorkflowBuilder` om sequentiële
   berichtoverdracht tussen agenten te modelleren.
4. **Hoe A2A in productie werkt** — het mogelijk maken van cross-framework-, cross-service-samenwerking
   met dynamische ontdekking en streaming-updates.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
Disclaimer:
Dit document is vertaald met behulp van de AI-vertalingsdienst Co-op Translator (https://github.com/Azure/co-op-translator). Hoewel wij streven naar nauwkeurigheid, dient u zich ervan bewust te zijn dat geautomatiseerde vertalingen fouten of onnauwkeurigheden kunnen bevatten. Het oorspronkelijke document in de oorspronkelijke taal moet als de gezaghebbende bron worden beschouwd. Voor kritieke informatie wordt professionele menselijke vertaling aanbevolen. Wij zijn niet aansprakelijk voor eventuele misverstanden of verkeerde interpretaties die voortvloeien uit het gebruik van deze vertaling.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
